In [ ]:
import pandas as pd

## Datasets

#### Pools and Reviews

In [ ]:
## swimming pools data

pools_df = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/swimmingpool_list_to_use_25112025.csv")
pools_df.drop(columns=['status', 'cluster_id', 'index_right', 'Unnamed: 54', 'Unnamed: 55', 'match_score', 'match_type'], inplace=True)
pools_df.info()

In [ ]:
## all reviews
reviews_df= pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/all_collected_reviews_25112025.parquet")
reviews_df.info()

In [ ]:
### review summarries
summaries = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/text_reviews_summarized_25112025.parquet")
summaries.info()

In [ ]:
##those reviews that were classified
classed_reviews_df = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/classified_reviews_05122025.parquet")
classed_reviews_df.info()

In [ ]:
classed_reviews_df.social_encounter.value_counts()

In [ ]:
classed_reviews_df = classed_reviews_df[(classed_reviews_df['classification_status'] != 'FAILED') &
                                        (classed_reviews_df['social_encounter'] != 'MISSING')]
#if there are any rows where the racial connotation is yes but the social encounter is no change the social encounter to yes
classed_reviews_df.loc[(classed_reviews_df['racial_connotations']=='yes') &
                       (classed_reviews_df['social_encounter']=='no'), 'social_encounter'] = 'yes'
#if the racial connotation is no then the connotation type must be N/A
classed_reviews_df.loc[classed_reviews_df['racial_connotations']=='no', 'connotation_type'] = 'N/A'
#if  the social and racial connotation are both yes and the connotation type is  neutral change it to ambiguous
classed_reviews_df.loc[(classed_reviews_df['social_encounter']=='yes') &
                       (classed_reviews_df['racial_connotations']=='yes') &
                       (classed_reviews_df['connotation_type']=='neutral'), 'connotation_type'] = 'ambiguous'
classed_reviews_df.loc
classed_reviews_df.info()

In [ ]:
classed_reviews_df[classed_reviews_df['was_summarized'].isnull()]

In [ ]:
classed_reviews_df[classed_reviews_df['classification_status'] =='SUCCESS'].nunique()

In [ ]:
# use the existing dataframe variable defined earlier
classed_reviews_df.was_summarized.value_counts()

In [ ]:
classed_reviews_df[(classed_reviews_df['social_encounter']=='no') &
                   (classed_reviews_df['racial_connotations'] =='yes')]

In [ ]:
classed_reviews_df.connotation_type.value_counts()

In [ ]:
classed_reviews_df[classed_reviews_df['social_encounter']== 'MISSING']

### Data cleaning and processing

In [ ]:
### change all column names to lowercase
pools_df.columns = pools_df.columns.str.lower()
reviews_df.columns = reviews_df.columns.str.lower()
summaries.columns = summaries.columns.str.lower()
classed_reviews_df.columns = classed_reviews_df.columns.str.lower()

In [ ]:
pools_df.columns

In [ ]:
## select columns of interest from each dataframe

pools_df_cols = ['place_id', 'owner', 'address_gmaps', 'latitude_gmaps', 'longitude_gmaps', 'place_name',
                  'total_reviews', 'place_rating', 'primary_place_type', 'place_subtype',
                  'geometry', 'wkr_nr', 'wkr_name', 'land_nr', 'land_name']

reviews_df_cols = ['place_id', 'reviewer_id', 'reviewer_name', 'review_id', 'review_link',
       'contributor_link', 'review_time', 'org_review', 'org_review_lang',
       'trans_review', 'reviewer_rating', 'other_reviews', 'uploaded_pics']

classed_reviews_df_cols = ['review_id', 'social_encounter','racial_connotations',
                           'connotation_type', 'classification_status']

In [ ]:
## merged dataframe for analysis
merged_df = reviews_df[reviews_df_cols].merge(classed_reviews_df[classed_reviews_df_cols], on=['review_id'],  how='inner')\
.merge(pools_df[pools_df_cols], on='place_id', how='inner')
# .merge(
    # summaries, on=['place_id', 'review_id'], how='left')
    
##rename columns
merged_df.rename(columns={'wrk_nr':'constituency_number','wkr_name':'constituency' }, inplace=True)
merged_df.info()

In [ ]:
## dropping the gmaps suffix in columns
merged_df.rename(columns={'address_gmaps':'address',
                          'latitude_gmaps':'latitude',
                          'longitude_gmaps':'longitude'}, inplace=True)

In [ ]:
merged_df[merged_df['connotation_type'].isin(['unpleasant', 'neutral'])][['review_id', 'trans_review','summary', 'social_encounter', 'racial_connotations', 'connotation_type']]

#### Demographic and voters share

In [ ]:
voters_df= pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/voters_share - second_vote_constituency.csv")
##rename columns to lowercase
voters_df.columns = voters_df.columns.str.lower()
voters_df.rename(columns={'state_name':'land_name', 'state_code':'land_code'}, inplace=True)
voters_df.info()

In [ ]:
demographics_df= pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/constituency_strukturdaten_28112025.csv")
## change column names for easier merging later
demographics_df.rename(columns={'Land':'land_name', 'constituency_name': 'constituency'}, inplace=True)
demographics_df.columns = demographics_df.columns.str.lower()
demographics_df.info()

### Merging all datasets

In [ ]:
rvd_merged_df = merged_df.merge(voters_df, on=['land_name', 'constituency'], how='left').merge(
    demographics_df, on=['land_name', 'constituency'], how='left')
rvd_merged_df.info()

### Name to ethnicity

In [ ]:
## merge with the name ethnicity
name_ethnicity = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/name_ethnicity_prediction/name_ethnicity_prediction.csv")
name_ethnicity.info()

In [ ]:
# Drop the 'Unnamed: 0' column
name_ethnicity = name_ethnicity.drop(columns=['Unnamed: 0'])

# Drop the first row
name_ethnicity = name_ethnicity.drop(index=0)

# Split the 'demography' column into three new columns
name_ethnicity[['ethnicity', 'age', 'gender']] = name_ethnicity['demographic'].str.split('_', expand=True)
name_ethnicity = name_ethnicity.drop(columns=['demographic'])
name_ethnicity = name_ethnicity.rename(columns={'name': 'reviewer_name'})
# Display the updated DataFrame
name_ethnicity.head()

In [ ]:
# !pip install pycountry

In [ ]:
import pycountry

# Function to convert alpha-2 code to full country name with additional checks
def get_country_name(alpha_2_code):
    if pd.isna(alpha_2_code):
        return None
    try:
        return pycountry.countries.get(alpha_2=alpha_2_code.upper()).name
    except (AttributeError, KeyError):
        return None

# Perform the merge
rvd_eth_merged_df = pd.merge(rvd_merged_df, name_ethnicity, on='reviewer_name', how='inner')

# Drop duplicates
rvd_eth_merged_df = rvd_eth_merged_df.drop_duplicates().reset_index(drop=True)

# Apply the function to create the nationality column in rvd_eth_merged_df DataFrame
rvd_eth_merged_df['nationality'] = rvd_eth_merged_df['ethnicity'].apply(get_country_name)

# Display the resulting DataFrame
display(rvd_eth_merged_df[['reviewer_name', 'ethnicity', 'nationality']])

In [ ]:
rvd_eth_merged_df.to_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/final_merged_data_for_analysis_03122025.parquet")

# data agumentations and manipulations


#### We need to create a new pool type column

In [ ]:
# for rows with pool_id, group by pool_id and get unique values and their counts for primary_place_type
sp_pids_pool_types = pools_df[['pool_id', 'pool_type', 'primary_place_type']].dropna(subset=['pool_id'])

# unique primary_place_type values per pool_type (returns ndarray for each group)
unique_by_pid = sp_pids_pool_types.groupby('pool_type')['primary_place_type'].unique()

# number of unique primary_place_type values per pool_type
unique_counts = sp_pids_pool_types.groupby('pool_type')['primary_place_type'].nunique()

# display a sample of results
print(unique_by_pid.head())
print(unique_counts.head())

In [ ]:
#shapefile data
import geopandas as gpd
shapefile_con = gpd.read_file("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/btw25_geometrie_wahlkreise_shp/btw25_geometrie_wahlkreise_shp.shp")
#change column names
shapefile_con.rename(columns={'WKR_NAME':'constituency_name', 'WKR_NR':'constituency_number',
                              'LAND_NAME':'land_name','LAND_NR':'land_number'}, inplace=True)
shapefile_con.info()

#### Trying to map  those on goolge to the others would be tricky we are just going to give then a name

In [ ]:
### for rows where the pool type is  missing but place_id is present we fill in with 'on-google'
pools_df['pool_type'] = pools_df.apply(
    lambda row: 'on-google' if pd.isna(row['pool_type']) and pd.notna(row['place_id']) else row['pool_type'],
    axis=1
)
pools_df.info()

In [ ]:
pools_df.WKR_NAME.nunique()

In [ ]:
pools_qgis=pools_df[['place_id', 'place_name', 'pool_type', 'latitude_gmaps', 'longitude_gmaps', 
          'total_reviews', 'place_rating', 'city_admin']]

In [ ]:
pools_gdf = gpd.GeoDataFrame(
    pools_qgis, 
    geometry=gpd.points_from_xy(pools_qgis.longitude_gmaps, pools_qgis.latitude_gmaps),
    crs="EPSG:4326"
)
# Ensure both GeoDataFrames use the same CRS before spatial join
if shapefile_con.crs != pools_gdf.crs:
    shapefile_con_reproj = shapefile_con.to_crs(pools_gdf.crs)
else:
    shapefile_con_reproj = shapefile_con


# First try within join
pools_within = gpd.sjoin(pools_gdf, shapefile_con_reproj, how="inner", predicate='within')

# For points that didn't match, try intersects
remaining_points = pools_gdf[~pools_gdf.index.isin(pools_within.index)]
pools_intersects = gpd.sjoin(remaining_points, shapefile_con_reproj, how="inner", predicate='intersects')

# Combine results
pools_qgis_gdf = pd.concat([pools_within, pools_intersects], ignore_index=True)

# If there are still unmatched points, do a nearest join as last resort
if len(pools_qgis_gdf) < len(pools_gdf):
    remaining_points = pools_gdf[~pools_gdf.index.isin(pools_qgis_gdf.index)]
    pools_nearest = gpd.sjoin_nearest(remaining_points, shapefile_con_reproj, how="left", max_distance=1000)
    pools_qgis_gdf = pd.concat([pools_qgis_gdf, pools_nearest], ignore_index=True)

pools_qgis_gdf.info()    

In [ ]:
final_pools_qgis = pd.merge( pools_qgis_gdf, demographics_df, on=['constituency_number',
                            'constituency_name','land_name' ], how='right')
final_pools_qgis.drop(columns=['index_right'], inplace=True)
final_pools_qgis.info()

In [ ]:
final_pools_qgis.to_file("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/pools_qgis_data/pools_qgis_data.shp")